In [32]:
# %% 1. Imports and setup
import pandas as pd
import pyreadstat
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"C:\Users\Hp\Downloads\Project 2026 DS")# %% 2. Shared definitions
demo_cols = ['Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3', 'LondInOut',
             'NSSEC5', 'Educ6', 'Orient4', 'Relig7', 'ChildAgeU13', 'Maternity_pop']
group_cols = ['LA'] + demo_cols

imd_map = {i: f"Decile {i}" for i in range(1, 11)}
imd_map[1] = "Decile 1 (most deprived)"
imd_map[10] = "Decile 10 (least deprived)"

london_boroughs = [
    'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden',
    'City of London', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney',
    'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon',
    'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames',
    'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames',
    'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

covid_years = ['2020-21', '2021-22']

likert_map = {
    'Strongly disagree': 1,
    'Disagree': 2,
    'Neither agree nor disagree': 3,
    'Agree': 4,
    'Strongly agree': 5,
    'Not asked: done 150 mins of moderate activity in last week': np.nan
}

def clean_la_name(name):
    if pd.isna(name):
        return name
    return re.sub(r'^[EWS]\d{8}\s+', '', name)

def clean_readiness(series):
    return series.map(likert_map)

def weighted_avg(series, weights):
    d = pd.DataFrame({'v': series, 'w': weights}).dropna()
    if d.empty or d['w'].sum() == 0:
        return np.nan
    return np.average(d['v'], weights=d['w'])

In [33]:
# %% process_year - updated to accept the correct readiness_opportunity column name per year
def process_year(year_label, filepath, cols_needed, la_col, readyop_col):
    df, meta = pyreadstat.read_sav(filepath, usecols=cols_needed, apply_value_formats=True)

    df[la_col] = df[la_col].astype(str).apply(clean_la_name)
    lon = df[df[la_col].isin(london_boroughs)].copy()
    lon = lon.rename(columns={la_col: 'LA'})

    if pd.api.types.is_categorical_dtype(lon['LA']):
        lon['LA'] = lon['LA'].cat.remove_unused_categories()

    print(f"{year_label}: {len(lon)} London respondents, {lon['LA'].nunique()} boroughs")

    for col in group_cols:
        lon[col] = lon[col].astype(object).fillna('Not asked / Not applicable').astype(str)

    imd_raw, _ = pyreadstat.read_sav(filepath, usecols=['serial', 'IMD10'], apply_value_formats=False)
    imd_raw = imd_raw.rename(columns={'IMD10': 'IMD10_raw'})
    lon = lon.merge(imd_raw, on='serial', how='left')
    lon['IMD10_numeric'] = lon['IMD10_raw']
    lon['IMD10'] = lon['IMD10_raw'].map(imd_map).fillna('Not asked / Not applicable')
    lon = lon.drop(columns=['IMD10_raw'])

    lon['Filter_Act'] = lon['Filter_Act'].astype(float)
    lon['Filter_InsAct'] = lon['Filter_InsAct'].astype(float)
    lon['Filter_Inact'] = lon['Filter_Inact'].astype(float)
    lon['wt_final'] = lon['wt_final'].astype(float)

    if 'MEMS7_ALL' in lon.columns:
        lon['MEMS7_ALL'] = pd.to_numeric(lon['MEMS7_ALL'], errors='coerce')

    # standardise the readiness_opportunity column name to 'READYOP1_POP' regardless of year
    if readyop_col in lon.columns:
        lon['READYOP1_POP'] = clean_readiness(lon[readyop_col])
    if 'READYAB1_POP' in lon.columns:
        lon['READYAB1_POP'] = clean_readiness(lon['READYAB1_POP'])

    gap = lon.groupby('LA').apply(lambda x: pd.Series({
        'pct_active': weighted_avg(x['Filter_Act'], x['wt_final']) * 100,
        'pct_fairly_active': weighted_avg(x['Filter_InsAct'], x['wt_final']) * 100,
        'pct_inactive': weighted_avg(x['Filter_Inact'], x['wt_final']) * 100,
        'mems7_all_wtd': weighted_avg(x['MEMS7_ALL'], x['wt_final']) if 'MEMS7_ALL' in x.columns else np.nan,
        'readiness_ability_wtd': weighted_avg(x['READYAB1_POP'], x['wt_final']) if 'READYAB1_POP' in x.columns else np.nan,
        'readiness_opportunity_wtd': weighted_avg(x['READYOP1_POP'], x['wt_final']) if 'READYOP1_POP' in x.columns else np.nan,
        'respondents': len(x),
        'weighted_base': x['wt_final'].sum()
    })).reset_index()

    gap = gap.rename(columns={'LA': 'borough'})
    gap['survey_year'] = year_label
    gap['covid_affected'] = year_label in covid_years
    gap['readyop_source_var'] = readyop_col  # track which raw variable was used, for transparency
    gap['source_file'] = os.path.basename(filepath)

    # (population profile section unchanged from before)
    rows = []
    for col in demo_cols:
        temp = lon.groupby(['LA', col]).apply(lambda x: pd.Series({
            'pct_active': weighted_avg(x['Filter_Act'], x['wt_final']) * 100,
            'pct_fairly_active': weighted_avg(x['Filter_InsAct'], x['wt_final']) * 100,
            'pct_inactive': weighted_avg(x['Filter_Inact'], x['wt_final']) * 100,
            'respondents': len(x),
            'weighted_base': x['wt_final'].sum()
        })).reset_index()
        temp = temp.rename(columns={'LA': 'borough', col: 'category'})
        temp['demographic_group'] = col
        rows.append(temp)

    london_wide_rows = []
    for col in demo_cols:
        temp = lon.groupby(col).apply(lambda x: pd.Series({
            'pct_active': weighted_avg(x['Filter_Act'], x['wt_final']) * 100,
            'pct_fairly_active': weighted_avg(x['Filter_InsAct'], x['wt_final']) * 100,
            'pct_inactive': weighted_avg(x['Filter_Inact'], x['wt_final']) * 100,
            'respondents': len(x),
            'weighted_base': x['wt_final'].sum()
        })).reset_index()
        temp = temp.rename(columns={col: 'category'})
        temp['borough'] = 'London-wide'
        temp['demographic_group'] = col
        london_wide_rows.append(temp)

    profile = pd.concat(rows + london_wide_rows, ignore_index=True)
    profile['survey_year'] = year_label
    profile['suppress'] = profile['respondents'] < 30
    profile['source_file'] = os.path.basename(filepath)
    profile['geography_level'] = profile['borough'].apply(lambda b: 'London-wide' if b == 'London-wide' else 'borough')

    return gap, profile

In [34]:
# %% 4. File paths for all 7 years
# %% Updated all_files dict - now includes the correct READYOP variable name per year
all_files = {
    "2016-17": (r"ActiveLives_Data\surveydata1617.sav", "LA", "READYOP1_POP"),
    "2017-18": (r"ActiveLives_Data\surveydata1718.sav", "LA", "READYOP1_POP"),
    "2018-19": (r"ActiveLives_Data\surveydata1819.sav", "LA_2023", "READYOP1_POP"),
    "2019-20": (r"ActiveLives_Data\surveydata1920.sav", "LA_2023", "READYOP_CV_3_POP"),
    "2020-21": (r"ActiveLives_Data\surveydata2021.sav", "LA_2023", "READYOP_CV_3_POP"),
    "2021-22": (r"ActiveLives_Data\surveydata2122.sav", "LA_2023", "READYOP1_POP"),
    "2022-23": (r"ActiveLives_Data\surveydata2223.sav", "LA_2023", "READYOP1_POP"),
}

cols_base = ['serial', 'wt_final', 'Reg9', 'LondInOut',
             'Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3',
             'NSSEC5', 'Educ6', 'Orient4', 'Relig7',
             'ChildAgeU13', 'Maternity_pop',
             'Filter_Act', 'Filter_InsAct', 'Filter_Inact']

extra_vars = ['MEMS7_ALL', 'READYAB1_POP', 'READYOP1_POP']

In [35]:
# %% Re-run processing loop with the corrected readyop column mapping
gap_results = []
profile_results = []

for year, (filepath, la_col, readyop_col) in all_files.items():
    _, meta = pyreadstat.read_sav(filepath, metadataonly=True)
    extra_present = [v for v in ['MEMS7_ALL', 'READYAB1_POP'] if v in meta.column_names]
    if readyop_col in meta.column_names:
        extra_present.append(readyop_col)
    cols_needed = cols_base + [la_col] + extra_present

    gap_y, profile_y = process_year(year, filepath, cols_needed, la_col, readyop_col)
    gap_results.append(gap_y)
    profile_results.append(profile_y)

gapscore_v2 = pd.concat(gap_results, ignore_index=True)
gapscore_v2 = gapscore_v2[['survey_year', 'borough', 'pct_active', 'pct_fairly_active', 'pct_inactive',
                            'mems7_all_wtd', 'readiness_ability_wtd', 'readiness_opportunity_wtd',
                            'respondents', 'weighted_base', 'covid_affected', 'readyop_source_var', 'source_file']]

poplprofile_v2 = pd.concat(profile_results, ignore_index=True)
poplprofile_v2 = poplprofile_v2[['survey_year', 'geography_level', 'borough', 'demographic_group', 'category',
                                  'pct_active', 'pct_fairly_active', 'pct_inactive',
                                  'respondents', 'weighted_base', 'suppress', 'source_file']]

gapscore_v2.to_csv(r"ActiveLives_Data\gapscore_v2.csv", index=False)
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv", index=False)

print(gapscore_v2[['survey_year', 'readyop_source_var', 'readiness_opportunity_wtd']].groupby(['survey_year', 'readyop_source_var']).mean())

2016-17: 19497 London respondents, 33 boroughs
2017-18: 16200 London respondents, 33 boroughs
2018-19: 16148 London respondents, 33 boroughs
2019-20: 16364 London respondents, 33 boroughs
2020-21: 16340 London respondents, 33 boroughs
2021-22: 16382 London respondents, 33 boroughs
2022-23: 16748 London respondents, 33 boroughs
                                readiness_opportunity_wtd
survey_year readyop_source_var                           
2016-17     READYOP1_POP                         3.730421
2017-18     READYOP1_POP                         3.754850
2018-19     READYOP1_POP                         4.085554
2019-20     READYOP_CV_3_POP                     4.120391
2020-21     READYOP_CV_3_POP                     4.092390
2021-22     READYOP1_POP                         4.036883
2022-23     READYOP1_POP                         4.063810


In [36]:
# %% 6. Combine and save final files
gapscore_v2 = pd.concat(gap_results, ignore_index=True)
gapscore_v2 = gapscore_v2[['survey_year', 'borough', 'pct_active', 'pct_fairly_active', 'pct_inactive',
                            'mems7_all_wtd', 'readiness_ability_wtd', 'readiness_opportunity_wtd',
                            'respondents', 'weighted_base', 'covid_affected', 'source_file']]

poplprofile_v2 = pd.concat(profile_results, ignore_index=True)
poplprofile_v2 = poplprofile_v2[['survey_year', 'geography_level', 'borough', 'demographic_group', 'category',
                                  'pct_active', 'pct_fairly_active', 'pct_inactive',
                                  'respondents', 'weighted_base', 'suppress', 'source_file']]

gapscore_v2.to_csv(r"ActiveLives_Data\gapscore_v2.csv", index=False)
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv", index=False)

print("gapscore_v2.csv:", gapscore_v2.shape)
print(gapscore_v2.head())
print("poplprofile_v2.csv:", poplprofile_v2.shape)

gapscore_v2.csv: (231, 12)
  survey_year               borough  pct_active  pct_fairly_active  \
0     2016-17  Barking and Dagenham   49.230649          16.065856   
1     2016-17                Barnet   56.778087          12.280310   
2     2016-17                Bexley   56.715084          17.347982   
3     2016-17                 Brent   55.859792          11.182239   
4     2016-17               Bromley   69.016348          11.295237   

   pct_inactive  mems7_all_wtd  readiness_ability_wtd  \
0     34.703494     586.890898               4.017710   
1     30.941603     582.438459               3.980225   
2     25.936934     593.441685               3.902070   
3     32.957969     563.870208               4.039761   
4     19.688415     765.689477               3.757762   

   readiness_opportunity_wtd  respondents  weighted_base  covid_affected  \
0                   3.684420        958.0     664.418228           False   
1                   3.824085        997.0    1324.609800 

In [37]:
# %% 7. Sanity check the readiness values now look correct (1-5 scale, not all NaN)
print(gapscore_v2[['survey_year', 'readiness_ability_wtd', 'readiness_opportunity_wtd']].groupby('survey_year').mean())

             readiness_ability_wtd  readiness_opportunity_wtd
survey_year                                                  
2016-17                   3.998417                   3.730421
2017-18                   4.001837                   3.754850
2018-19                   4.266670                   4.085554
2019-20                   4.262689                   4.120391
2020-21                   4.222851                   4.092390
2021-22                   4.204227                   4.036883
2022-23                   4.219045                   4.063810


In [38]:
# %% Quick check that 2016-17's larger sample isn't a duplication artifact
print(gapscore_v2[gapscore_v2['survey_year'] == '2016-17'][['respondents']].sum())
print(gapscore_v2[gapscore_v2['survey_year'] == '2017-18'][['respondents']].sum())

respondents    19497.0
dtype: float64
respondents    16200.0
dtype: float64


In [39]:
# %% Break down where your row count actually comes from
print("Total rows:", len(poplprofile_v2))
print()
print("Rows by geography_level:")
print(poplprofile_v2['geography_level'].value_counts())
print()
print("Rows per year:")
print(poplprofile_v2.groupby('survey_year').size())
print()
print("Rows per demographic_group:")
print(poplprofile_v2.groupby('demographic_group').size())
print()
print("Categories per demographic_group:")
print(poplprofile_v2.groupby('demographic_group')['category'].nunique())

Total rows: 14918

Rows by geography_level:
geography_level
borough        14449
London-wide      469
Name: count, dtype: int64

Rows per year:
survey_year
2016-17    2097
2017-18    2105
2018-19    2112
2019-20    2141
2020-21    2145
2021-22    2143
2022-23    2175
dtype: int64

Rows per demographic_group:
demographic_group
Age9             2139
ChildAgeU13       476
Disab3            952
Educ6            1661
Eth7             1901
Gend3             802
IMD10            2087
LondInOut         245
Maternity_pop     476
NSSEC5           1190
Orient4          1097
Relig7           1892
dtype: int64

Categories per demographic_group:
demographic_group
Age9              9
ChildAgeU13       2
Disab3            4
Educ6             7
Eth7              9
Gend3             4
IMD10            10
LondInOut         4
Maternity_pop     2
NSSEC5            7
Orient4           5
Relig7            9
Name: category, dtype: int64


In [40]:
# %% Compare demographic groups between old and new files, with row counts per variable
old_profile = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\poplprofile_merged_ROOT_BACKUP.csv")

print("--- OLD file: demographic_variable counts ---")
print(old_profile['demographic_variable'].value_counts())
print()
print("--- NEW file: demographic_group counts ---")
print(poplprofile_v2['demographic_group'].value_counts())
print()

old_vars = set(old_profile['demographic_variable'].unique())
new_vars = set(poplprofile_v2['demographic_group'].unique())

print("In OLD but not NEW:", old_vars - new_vars)
print("In NEW but not OLD:", new_vars - old_vars)
print("In both:", old_vars & new_vars)

--- OLD file: demographic_variable counts ---
demographic_variable
IMD10            1927
Age9             1851
Eth7             1475
Educ6            1291
Relig7           1276
NSSEC5           1023
Disab3            780
Orient4           696
Gend3             637
ChildAgeU13       474
Maternity_pop     344
LondInOut         245
Name: count, dtype: int64

--- NEW file: demographic_group counts ---
demographic_group
Age9             2139
IMD10            2087
Eth7             1901
Relig7           1892
Educ6            1661
NSSEC5           1190
Orient4          1097
Disab3            952
Gend3             802
ChildAgeU13       476
Maternity_pop     476
LondInOut         245
Name: count, dtype: int64

In OLD but not NEW: set()
In NEW but not OLD: set()
In both: {'Age9', 'Educ6', 'IMD10', 'Relig7', 'NSSEC5', 'ChildAgeU13', 'LondInOut', 'Eth7', 'Disab3', 'Gend3', 'Orient4', 'Maternity_pop'}


In [41]:
# %% Confirm demo_cols currently in use matches expected 12
print(demo_cols)
print(len(demo_cols))

['Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3', 'LondInOut', 'NSSEC5', 'Educ6', 'Orient4', 'Relig7', 'ChildAgeU13', 'Maternity_pop']
12


In [42]:
# %% Direct proof: total respondents captured, old vs new, for a 2018-23 year (where the bug existed)
old_2018 = old_profile[(old_profile['survey_year']=='2018-19') & (old_profile['demographic_variable']=='Age9')]
new_2018 = poplprofile_v2[(poplprofile_v2['survey_year']=='2018-19') & (poplprofile_v2['demographic_group']=='Age9')]

print("OLD 2018-19 Age9 total respondents:", old_2018['sample_size'].sum())
print("NEW 2018-19 Age9 total respondents:", new_2018['respondents'].sum())

OLD 2018-19 Age9 total respondents: 31156.0
NEW 2018-19 Age9 total respondents: 32296.0


In [43]:
# %% Check how much of the data is "Not asked / Not applicable"
not_asked = poplprofile_v2[poplprofile_v2['category'] == 'Not asked / Not applicable']
print(f"{len(not_asked)} rows out of {len(poplprofile_v2)} total ({len(not_asked)/len(poplprofile_v2)*100:.1f}%)")
print(not_asked['demographic_group'].value_counts())

1787 rows out of 14918 total (12.0%)
demographic_group
Disab3     238
Eth7       238
Relig7     238
Orient4    238
Educ6      237
Age9       236
NSSEC5     204
Gend3      158
Name: count, dtype: int64


In [44]:
# %% Find where "Not asked" rows are concentrated - by year and by borough
not_asked = poplprofile_v2[poplprofile_v2['category'] == 'Not asked / Not applicable']

print("By survey_year:")
print(not_asked['survey_year'].value_counts())
print()
print("By borough:")
print(not_asked['borough'].value_counts().head(10))
print()
print("By geography_level:")
print(not_asked['geography_level'].value_counts())

By survey_year:
survey_year
2019-20    263
2022-23    261
2017-18    259
2020-21    259
2018-19    258
2021-22    258
2016-17    229
Name: count, dtype: int64

By borough:
borough
Lewisham       55
London-wide    55
Wandsworth     54
Sutton         54
Ealing         54
Hounslow       54
Islington      54
Newham         54
Harrow         54
Haringey       54
Name: count, dtype: int64

By geography_level:
geography_level
borough        1732
London-wide      55
Name: count, dtype: int64


In [45]:
# %% Split poplprofile_v2 into two files: clean version + not-applicable-only version

# 1. The "Not asked / Not applicable" rows, saved separately
not_applicable = poplprofile_v2[poplprofile_v2['category'] == 'Not asked / Not applicable'].copy()
not_applicable.to_csv(r"ActiveLives_Data\poplprofile_v2_notapplicable.csv", index=False)

# 2. The clean version, with those rows removed (for use in charts/report tables)
poplprofile_v2_clean = poplprofile_v2[poplprofile_v2['category'] != 'Not asked / Not applicable'].copy()
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv", index=False)

print("Saved poplprofile_v2_notapplicable.csv:", not_applicable.shape)
print("Saved poplprofile_v2_clean.csv:", poplprofile_v2_clean.shape)
print()
print("Original poplprofile_v2.csv left untouched:", poplprofile_v2.shape)

Saved poplprofile_v2_notapplicable.csv: (1787, 12)
Saved poplprofile_v2_clean.csv: (13131, 12)

Original poplprofile_v2.csv left untouched: (14918, 12)


In [46]:
# %% Compare old file vs new clean file - standardise column names first
old_compare = old_profile.rename(columns={
    'demographic_variable': 'demographic_group',
    'demographic_category': 'category',
    'sample_size': 'respondents'
})[['survey_year', 'borough', 'demographic_group', 'category',
    'pct_active', 'pct_fairly_active', 'pct_inactive', 'respondents']]

new_compare = poplprofile_v2_clean[['survey_year', 'borough', 'demographic_group', 'category',
                                     'pct_active', 'pct_fairly_active', 'pct_inactive', 'respondents']]

# %% Find combinations present in one file but not the other
old_keys = set(zip(old_compare['survey_year'], old_compare['borough'], old_compare['demographic_group'], old_compare['category']))
new_keys = set(zip(new_compare['survey_year'], new_compare['borough'], new_compare['demographic_group'], new_compare['category']))

only_in_old = old_keys - new_keys
only_in_new = new_keys - old_keys

print(f"Rows only in OLD file: {len(only_in_old)}")
print(f"Rows only in NEW (clean) file: {len(only_in_new)}")
print()
print("Sample of rows only in NEW file (recovered by the borough-matching fix):")
for k in list(only_in_new)[:15]:
    print(k)

Rows only in OLD file: 1255
Rows only in NEW (clean) file: 2367

Sample of rows only in NEW file (recovered by the borough-matching fix):
('2019-20', 'Kingston upon Thames', 'Orient4', 'Bisexual')
('2019-20', 'Merton', 'Relig7', 'No religion')
('2019-20', 'Barnet', 'Orient4', 'Bisexual')
('2019-20', 'Bexley', 'IMD10', 'Decile 1 (most deprived)')
('2018-19', 'Lewisham', 'Orient4', 'Bisexual')
('2018-19', 'Enfield', 'Relig7', 'Jewish')
('2022-23', 'London-wide', 'Age9', '16-24')
('2018-19', 'Hammersmith and Fulham', 'Orient4', 'Bisexual')
('2018-19', 'Newham', 'Educ6', 'Level 1 and below')
('2019-20', 'City of London', 'Relig7', 'No religion')
('2019-20', 'Ealing', 'LondInOut', 'E13000002 Outer London')
('2018-19', 'Enfield', 'Orient4', 'Other sexual orientation')
('2019-20', 'Havering', 'Orient4', 'Gay or Lesbian')
('2019-20', 'Tower Hamlets', 'Age9', '85+')
('2019-20', 'Redbridge', 'Eth7', 'Black')


In [47]:
# %% For combinations present in BOTH, check if the actual values differ
merged = old_compare.merge(
    new_compare,
    on=['survey_year', 'borough', 'demographic_group', 'category'],
    suffixes=('_old', '_new'),
    how='inner'
)

merged['pct_inactive_diff'] = (merged['pct_inactive_new'] - merged['pct_inactive_old']).abs()
merged['respondents_diff'] = merged['respondents_new'] - merged['respondents_old']

print("Rows matched in both files:", len(merged))
print()
print("Distribution of pct_inactive differences:")
print(merged['pct_inactive_diff'].describe())
print()
print("Rows with the biggest pct_inactive differences:")
print(merged.sort_values('pct_inactive_diff', ascending=False)[
    ['survey_year', 'borough', 'demographic_group', 'category', 'pct_inactive_old', 'pct_inactive_new', 'pct_inactive_diff', 'respondents_old', 'respondents_new']
].head(15))

Rows matched in both files: 10764

Distribution of pct_inactive differences:
count    10764.000000
mean         1.350020
std          3.448722
min          0.000000
25%          0.000000
50%          0.030220
75%          1.631828
max        100.000000
Name: pct_inactive_diff, dtype: float64

Rows with the biggest pct_inactive differences:
      survey_year               borough demographic_group  \
8949      2022-23  Barking and Dagenham              Age9   
7396      2021-22                 Brent            Relig7   
5837      2020-21               Croydon           Orient4   
6186      2020-21                Harrow              Eth7   
9298      2022-23        City of London             Gend3   
10411     2022-23  Richmond upon Thames             IMD10   
9886      2022-23              Hounslow              Age9   
8402      2021-22              Lewisham            Relig7   
8406      2021-22              Lewisham            Relig7   
7281      2021-22                Barnet         

In [48]:
# %% Confirm all these extreme-difference cells are correctly flagged for suppression
extreme_diff_rows = merged[merged['pct_inactive_diff'] > 30]
print("All have respondents < 30:", (extreme_diff_rows['respondents_new'] < 30).all())

All have respondents < 30: True


In [49]:
# %% Check: are the "only in old" rows also tiny/unreliable samples?
only_in_old_df = old_compare.set_index(['survey_year','borough','demographic_group','category']).loc[list(only_in_old)]
print(only_in_old_df['respondents'].describe())
print(only_in_old_df.sort_values('respondents', ascending=False).head(10))

count     1255.000000
mean      1165.340239
std       2550.970161
min          1.000000
25%         36.000000
50%        344.000000
75%        721.000000
max      18847.000000
Name: respondents, dtype: float64
                                                                  pct_active  \
survey_year borough demographic_group category                                 
2016-17     London  Maternity_pop     No                           62.489297   
2022-23     London  Maternity_pop     No                           66.600000   
2020-21     London  Maternity_pop     No                           65.300000   
2021-22     London  Maternity_pop     No                           66.800000   
2017-18     London  Maternity_pop     No                           64.652184   
2019-20     London  Maternity_pop     No                           65.400000   
2016-17     London  ChildAgeU13       No                           63.124468   
2018-19     London  Maternity_pop     No                           66.

In [50]:
# %% Confirm this is just a naming difference, not missing data
old_london_wide = old_compare[old_compare['borough'] == 'London']
new_london_wide = new_compare[new_compare['borough'] == 'London-wide']

print("OLD 'London' rows:", len(old_london_wide))
print("NEW 'London-wide' rows:", len(new_london_wide))

OLD 'London' rows: 430
NEW 'London-wide' rows: 414


In [51]:
# %% Fix LondInOut category labels - strip ONS region code prefix, same pattern as borough names
poplprofile_v2['category'] = poplprofile_v2.apply(
    lambda row: clean_la_name(row['category']) if row['demographic_group'] == 'LondInOut' else row['category'],
    axis=1
)

# re-save both the full and clean versions with this fix applied
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv", index=False)
poplprofile_v2_clean = poplprofile_v2[poplprofile_v2['category'] != 'Not asked / Not applicable'].copy()
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv", index=False)

print(poplprofile_v2[poplprofile_v2['demographic_group']=='LondInOut']['category'].unique())

['Outer London' 'Inner London']


In [52]:
# %% Check the full set of Eth7 categories in an old-style year vs a year with the new label
_, meta_old_era = pyreadstat.read_sav(r"ActiveLives_Data\surveydata1718.sav", metadataonly=True)
_, meta_new_era = pyreadstat.read_sav(r"ActiveLives_Data\surveydata2021.sav", metadataonly=True)

print("Eth7 value labels, 2017-18:")
print(meta_old_era.variable_value_labels.get('Eth7'))
print()
print("Eth7 value labels, 2020-21:")
print(meta_new_era.variable_value_labels.get('Eth7'))

Eth7 value labels, 2017-18:
{-99.0: 'Missing, should have been answered', -98.0: 'Not applicable: Survey routing', -97.0: 'Incorrectly multicoded', -96.0: 'Out of range', -95.0: "Cannot give an estimate / Don't know", -94.0: 'Prefer not to say', 1.0: 'White British', 2.0: 'White Other', 3.0: 'South Asian', 4.0: 'Black', 5.0: 'Chinese', 6.0: 'Mixed', 7.0: 'Other ethnic group'}

Eth7 value labels, 2020-21:
{-99.0: 'Missing, should have been answered', -98.0: 'Not applicable: Survey routing', -97.0: 'Incorrectly multicoded', -96.0: 'Out of range', -95.0: "Cannot give an estimate / Don't know", -94.0: 'Prefer not to say', 1.0: 'White British', 2.0: 'White Other', 3.0: 'Asian (excl. Chinese)', 4.0: 'Black', 5.0: 'Chinese', 6.0: 'Mixed', 7.0: 'Other ethnic group'}


In [53]:
# %% Standardise the Eth7 label - use "South Asian" consistently across all years
poplprofile_v2['category'] = poplprofile_v2['category'].replace('Asian (excl. Chinese)', 'South Asian')

# re-save
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv", index=False)
poplprofile_v2_clean = poplprofile_v2[poplprofile_v2['category'] != 'Not asked / Not applicable'].copy()
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv", index=False)

print(poplprofile_v2[poplprofile_v2['demographic_group']=='Eth7']['category'].unique())

['Black' 'Chinese' 'Mixed' 'Not asked / Not applicable'
 'Other ethnic group' 'South Asian' 'White British' 'White Other']


In [54]:
# %% Apply all fixes to poplprofile_v2_clean.csv

# 1. Fix LondInOut - strip ONS region code prefix
poplprofile_v2_clean['category'] = poplprofile_v2_clean.apply(
    lambda row: clean_la_name(row['category']) if row['demographic_group'] == 'LondInOut' else row['category'],
    axis=1
)

# 2. Standardise Eth7 label back to "South Asian"
poplprofile_v2_clean['category'] = poplprofile_v2_clean['category'].replace('Asian (excl. Chinese)', 'South Asian')

# re-save
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv", index=False)

print("LondInOut categories:", poplprofile_v2_clean[poplprofile_v2_clean['demographic_group']=='LondInOut']['category'].unique())
print("Eth7 categories:", poplprofile_v2_clean[poplprofile_v2_clean['demographic_group']=='Eth7']['category'].unique())
print()
print("Final shape:", poplprofile_v2_clean.shape)

LondInOut categories: ['Outer London' 'Inner London']
Eth7 categories: ['Black' 'Chinese' 'Mixed' 'Other ethnic group' 'South Asian'
 'White British' 'White Other']

Final shape: (13131, 12)


In [55]:
# %% Re-compare old file vs the now-fixed clean file
old_compare_fixed = old_profile.rename(columns={
    'demographic_variable': 'demographic_group',
    'demographic_category': 'category',
    'sample_size': 'respondents'
})[['survey_year', 'borough', 'demographic_group', 'category',
    'pct_active', 'pct_fairly_active', 'pct_inactive', 'respondents']]

old_compare_fixed['borough'] = old_compare_fixed['borough'].replace('London', 'London-wide')

new_compare_fixed = poplprofile_v2_clean[['survey_year', 'borough', 'demographic_group', 'category',
                                            'pct_active', 'pct_fairly_active', 'pct_inactive', 'respondents']]

# also drop "Not asked" from old file for a fair comparison, since clean file has it removed
old_compare_fixed = old_compare_fixed[old_compare_fixed['category'] != 'Not asked / Not applicable']

old_keys = set(zip(old_compare_fixed['survey_year'], old_compare_fixed['borough'], old_compare_fixed['demographic_group'], old_compare_fixed['category']))
new_keys = set(zip(new_compare_fixed['survey_year'], new_compare_fixed['borough'], new_compare_fixed['demographic_group'], new_compare_fixed['category']))

only_in_old = old_keys - new_keys
only_in_new = new_keys - old_keys

print(f"Rows only in OLD: {len(only_in_old)}")
print(f"Rows only in NEW: {len(only_in_new)}")
print()

if only_in_old:
    print("Sample only in OLD:")
    for k in sorted(only_in_old)[:10]:
        print(" ", k)

if only_in_new:
    print("Sample only in NEW:")
    for k in sorted(only_in_new)[:10]:
        print(" ", k)

Rows only in OLD: 57
Rows only in NEW: 1657

Sample only in OLD:
  ('2016-17', 'Barking and Dagenham', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Barnet', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Bexley', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Brent', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Bromley', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Camden', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'City of London', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Croydon', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Ealing', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
  ('2016-17', 'Enfield', 'NSSEC5', 'NS SEC 9: Students and other / unclassified')
Sample only in NEW:
  ('2016-17', 'Barking and Dagenham', 'NSSEC5', 'NS SEC 9: Students and other')
 

In [56]:
# %% Standardise NSSEC5 label - use the full version consistently across all years
poplprofile_v2_clean['category'] = poplprofile_v2_clean['category'].replace(
    'NS SEC 9: Students and other',
    'NS SEC 9: Students and other / unclassified'
)

poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv", index=False)

print(poplprofile_v2_clean[poplprofile_v2_clean['demographic_group']=='NSSEC5']['category'].unique())

['Aged <16 or 75+' 'NS SEC 1-2: Higher social groups'
 'NS SEC 3-5: Middle social groups' 'NS SEC 6-8: Lower social groups'
 'NS SEC 9: Students and other / unclassified']


In [57]:
# %% Re-check only_in_old count after this fix
old_keys = set(zip(old_compare_fixed['survey_year'], old_compare_fixed['borough'], old_compare_fixed['demographic_group'], old_compare_fixed['category']))
new_compare_final = poplprofile_v2_clean[['survey_year', 'borough', 'demographic_group', 'category',
                                            'pct_active', 'pct_fairly_active', 'pct_inactive', 'respondents']]
new_keys = set(zip(new_compare_final['survey_year'], new_compare_final['borough'], new_compare_final['demographic_group'], new_compare_final['category']))

only_in_old = old_keys - new_keys
print(f"Rows only in OLD (should now be near 0): {len(only_in_old)}")

Rows only in OLD (should now be near 0): 23


In [58]:
# %% Show exactly which 23 rows are still only in OLD, in your live notebook state
old_keys = set(zip(old_compare_fixed['survey_year'], old_compare_fixed['borough'], old_compare_fixed['demographic_group'], old_compare_fixed['category']))
new_compare_final = poplprofile_v2_clean[['survey_year', 'borough', 'demographic_group', 'category',
                                            'pct_active', 'pct_fairly_active', 'pct_inactive', 'respondents']]
new_keys = set(zip(new_compare_final['survey_year'], new_compare_final['borough'], new_compare_final['demographic_group'], new_compare_final['category']))

only_in_old = old_keys - new_keys
print(f"Remaining: {len(only_in_old)}")
for k in sorted(only_in_old):
    print(" ", k)

Remaining: 23
  ('2018-19', 'Barking and Dagenham', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Barnet', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Bexley', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Brent', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Bromley', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Camden', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Croydon', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Ealing', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Enfield', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Greenwich', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Hammersmith and Fulham', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Harrow', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Havering', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Hillingdon', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Hounslow', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Kensington and Chelsea', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19', 'Kingston upon Thames', 'NSSEC5', 'Aged <16 or 75+')
  ('2018-19'

In [59]:
# %% Check if "Aged <16 or 75+" exists at all in your new file for 2018-19
check = poplprofile_v2_clean[
    (poplprofile_v2_clean['survey_year'] == '2018-19') &
    (poplprofile_v2_clean['demographic_group'] == 'NSSEC5')
]
print(check['category'].value_counts())
print()
print("Boroughs with 'Aged <16 or 75+' in 2018-19 (new file):", 
      check[check['category']=='Aged <16 or 75+']['borough'].nunique())

category
NS SEC 1-2: Higher social groups               34
NS SEC 3-5: Middle social groups               34
NS SEC 6-8: Lower social groups                34
NS SEC 9: Students and other / unclassified    34
Name: count, dtype: int64

Boroughs with 'Aged <16 or 75+' in 2018-19 (new file): 0


In [60]:
# %% Go back to the raw 2018-19 .sav file directly and check NSSEC5 values for this category
df_check, meta_check = pyreadstat.read_sav(
    r"ActiveLives_Data\surveydata1819.sav",
    usecols=['LA_2023', 'NSSEC5'],
    apply_value_formats=True
)
df_check['LA_2023'] = df_check['LA_2023'].astype(str).apply(clean_la_name)
lon_check = df_check[df_check['LA_2023'].isin(london_boroughs)]

print(lon_check['NSSEC5'].value_counts(dropna=False))

NSSEC5
NS SEC 1-2: Higher social groups               9331
NS SEC 3-5: Middle social groups               2729
NS SEC 6-8: Lower social groups                1451
NS SEC 9: Students and other / unclassified    1407
NaN                                            1230
Name: count, dtype: int64


In [61]:
# %% Check the raw numeric NSSEC5 code for the NaN respondents in 2018-19
df_raw, meta_raw = pyreadstat.read_sav(
    r"ActiveLives_Data\surveydata1819.sav",
    usecols=['LA_2023', 'NSSEC5'],
    apply_value_formats=False   # get raw numeric codes, not labels
)
df_raw['LA_2023'] = df_raw['LA_2023'].astype(str).apply(clean_la_name)
lon_raw = df_raw[df_raw['LA_2023'].isin(london_boroughs)]

print(lon_raw['NSSEC5'].value_counts(dropna=False))
print()
print("Value labels available for NSSEC5 in 2018-19:")
print(meta_raw.variable_value_labels.get('NSSEC5'))

Series([], Name: count, dtype: int64)

Value labels available for NSSEC5 in 2018-19:
{-99.0: 'Missing, should have been answered', -98.0: 'Not applicable: Survey routing', -97.0: 'Incorrectly multicoded', -96.0: 'Out of range', -95.0: "Cannot give an estimate / Don't know", -94.0: 'Prefer not to say', 1.0: 'NS SEC 1-2: Higher social groups', 2.0: 'NS SEC 3-5: Middle social groups', 3.0: 'NS SEC 6-8: Lower social groups', 4.0: 'NS SEC 9: Students and other / unclassified', 5.0: 'Aged <16 or 75+'}


In [62]:
# %% Check Age9 categories are consistent across all years (fixed for 3-value all_files)
for year, (filepath, la_col, readyop_col) in all_files.items():
    df_check, meta_check = pyreadstat.read_sav(filepath, usecols=['Age9'], apply_value_formats=True)
    print(f"--- {year} ---")
    print(df_check['Age9'].value_counts(dropna=False))
    print()

--- 2016-17 ---
Age9
55-64    37130
65-74    37073
45-54    33400
35-44    30799
25-34    24414
75-84    15464
16-24    12688
85+       3842
NaN       1825
Name: count, dtype: int64

--- 2017-18 ---
Age9
65-74    34373
55-64    34267
45-54    30732
35-44    27709
25-34    22519
75-84    14236
16-24    10941
85+       3587
NaN       1383
Name: count, dtype: int64

--- 2018-19 ---
Age9
55-64    34745
65-74    33552
45-54    30982
35-44    28286
25-34    23335
75-84    14404
16-24    11329
85+       3311
NaN       1591
Name: count, dtype: int64

--- 2019-20 ---
Age9
55-64    33984
65-74    32426
45-54    29528
35-44    26921
25-34    23606
75-84    14626
16-24    11924
85+       3132
NaN       1588
Name: count, dtype: int64

--- 2020-21 ---
Age9
55-64    33895
65-74    32985
45-54    29145
35-44    26861
25-34    22800
75-84    15853
16-24    11143
85+       3651
NaN        940
Name: count, dtype: int64

--- 2021-22 ---
Age9
55-64    33487
65-74    32482
45-54    28069
35-44    27421
25-3

In [63]:
# %% Correct version - LA_2023 labelled (for filtering), NSSEC5 raw (to see the actual code)
df_mixed, meta_mixed = pyreadstat.read_sav(
    r"ActiveLives_Data\surveydata1819.sav",
    usecols=['LA_2023', 'NSSEC5'],
    apply_value_formats=True
)
# now separately pull NSSEC5 raw and merge back in by row position
df_raw_only, _ = pyreadstat.read_sav(
    r"ActiveLives_Data\surveydata1819.sav",
    usecols=['NSSEC5'],
    apply_value_formats=False
)
df_mixed['NSSEC5_raw'] = df_raw_only['NSSEC5']

df_mixed['LA_2023'] = df_mixed['LA_2023'].astype(str).apply(clean_la_name)
lon_mixed = df_mixed[df_mixed['LA_2023'].isin(london_boroughs)]

print(lon_mixed['NSSEC5_raw'].value_counts(dropna=False))

NSSEC5_raw
1.0    9331
2.0    2729
3.0    1451
4.0    1407
NaN    1230
Name: count, dtype: int64


In [64]:
# %% Save each year's column names + labels to a separate CSV file
import pyreadstat
import pandas as pd
import os

all_files_check = {
    "2016-17": r"ActiveLives_Data\surveydata1617.sav",
    "2017-18": r"ActiveLives_Data\surveydata1718.sav",
    "2018-19": r"ActiveLives_Data\surveydata1819.sav",
    "2019-20": r"ActiveLives_Data\surveydata1920.sav",
    "2020-21": r"ActiveLives_Data\surveydata2021.sav",
    "2021-22": r"ActiveLives_Data\surveydata2122.sav",
    "2022-23": r"ActiveLives_Data\surveydata2223.sav",
}

# make a folder to keep these organised
os.makedirs(r"ActiveLives_Data\column_lists", exist_ok=True)

for year, path in all_files_check.items():
    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    
    col_df = pd.DataFrame({
        'column': meta.column_names,
        'label': meta.column_labels
    })
    
    safe_year = year.replace("-", "_")
    outpath = fr"ActiveLives_Data\column_lists\columns_{safe_year}.csv"
    col_df.to_csv(outpath, index=False)
    print(f"{year}: saved {len(col_df)} columns to {outpath}")

2016-17: saved 7781 columns to ActiveLives_Data\column_lists\columns_2016_17.csv
2017-18: saved 8854 columns to ActiveLives_Data\column_lists\columns_2017_18.csv
2018-19: saved 9114 columns to ActiveLives_Data\column_lists\columns_2018_19.csv
2019-20: saved 10428 columns to ActiveLives_Data\column_lists\columns_2019_20.csv
2020-21: saved 10505 columns to ActiveLives_Data\column_lists\columns_2020_21.csv
2021-22: saved 10488 columns to ActiveLives_Data\column_lists\columns_2021_22.csv
2022-23: saved 10737 columns to ActiveLives_Data\column_lists\columns_2022_23.csv
